In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading the Hugging Face Transformer model...")
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set the pad token to the eos token to avoid padding warnings
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model loaded successfully!\n")
print("-" * 50)

def chat_with_bot():
    chat_history_ids = None
    step = 0

    print("Chatbot: Hello! I am your AI assistant. How can I help you today? (Type 'exit' or 'quit' to stop)")

    while True:
        user_input = input("User: ")

        if user_input.lower() in ['exit', 'quit']:
            print("Chatbot: Goodbye! Have a great day ahead.")
            break

        # Encode user input
        new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        # Append to chat history
        if step > 0:
            bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
        else:
            bot_input_ids = new_user_input_ids

        # Create the attention mask (1 for real tokens, 0 for padding)
        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        # Generate response with refined parameters
        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=1000,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=40,              # Lowered to make responses less random
            top_p=0.90,            # Lowered for better focus
            temperature=0.6        # Lowered to reduce sarcastic/wild outputs
        )

        # Decode the output
        bot_response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

        print(f"Chatbot: {bot_response}")
        step += 1

if __name__ == "__main__":
    chat_with_bot()